In [1]:
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime

In [7]:
conn = pyodbc.connect(
    'Driver={ODBC Driver 17 for SQL Server};'
    'Server=PUGSQLRSVRAG02.ug.sbicdirectory.com,5200;'
    'UID=RSVR_P_SQLACC_R_DS;'
    'Database=RSVRRealtimeFinacleUGPRD;'
    'PWD=M5tekitu45t65!g;'
    'Trusted_Connection=no;'
)

In [2]:
FEE_PATTERNS_JOIN = """
left join (select DISTINCT * FROM 
    (VALUES ('%FEE:%ATM%'),('%MANAGEMENT%FEE%'),('%MANAGEMENT%MONTHLY%FEE%'),('%MINIMUM%BALANCE%BREACH%FEE%'),('%FEE-CHEQUE%DEPOSIT%'),
    ('%BOL%FEES%'),('%BOL%CHARGES%'),('%CASH%DEPOSIT%FEE%'),('%ANNUAL%CARD%FEES%'),('%MGT%FEES%'),('%COMFEES%IFO%'),('%COMM%FEES%IFO'),('%INST MONEY TRAN CHARGE%'),
    ('%PAY%OFF%PENAL%INT%RECEIVABLE%'),('%PAYOFF%INT%RECEIVABLE%'),('%SI%EXECUTION-INTER%'),('%PENAL%P%&%L%INTEREST%'),('%PENAL%PL%INT%'),('%CASH WITHDRAWAL FEE%'),
    ('%INT%FOR%ACCOUNT%'),('%PENAL%INT.COLL%'),('%NORMAL%INT.COLL%'),('%SI%EXECUTION-NON%'),('%EXCISE%DUTY%'),('%Charge for OD limit%'),
    ('%EXCISE%DUTY%ON%ATW%'),('%STATEMENT%CHARGE%'),('%RTGS%CHARGE%'),('%DLG%CHARGE%'),('%COMM%CHARGE%'),('%MGT%FEE%'),('%COMM%FEE%'),
    ('%VISA%CARD%FEES%'),('%CLS%FEE%'),('%LIQUIDATION%FEE%'),('%REVERSAL%FEE'),('%SI%SETUP%FEE%'),('%URA%NET%CUSTODY%FEES'),
    ('%URA%NET%FEES%'),('%BOL%ENCASHMENT%CHG%INST%'),('%BOL%RENTAL%MONTHLY%FEE%'),('%MM%BOL%CHRGS'),('%CHGRE%STAMP%DUTY%'),
    ('%COMM%STAMP%DUTY%'),('%STAMP%FEE%'),('%STAMP%DUTY%IFO'),('%STAMP%DUTY%TAX%'),('%STAMP%DUTY%'),('%AGB%COMMISSION%'),('%AGENT%FEE%'),
    ('%AGB%EXCISE%'),('%BULKPAY%CHARGE%'),('%BULKMM%CHARGE%'),('%AGB%COMISION%'),('%FEE%'),('%SMS%CHARGE%'),('%ATW%CHARGE%'),('%COMMISSION%(FEE)%'),
    ('% WHT%'),('WHT%SWEEP%'),('Lease%Charge%'),('Access%Fee%'),('EXCISE%DUTY%'),('%CASH%WITHDRAWAL%FEE%'),
    ('%FEE%ACH%'),('%ACH%FEE%'),('%CARD%FEE%'),('%SBPOS%Commission%'),('%BANK%Commission%'),('%WTax%Pd%')) AS X(FEE_TXN)) N
    ON (hd.TRAN_PARTICULAR) LIKE N.FEE_TXN
"""

In [4]:
FRONTIER_ACID = 'UG6246002'

In [5]:
q1 = f"""
select hd.TRAN_DATE, hd.TRAN_ID, hd.PART_TRAN_SRL_NUM, hd.TRAN_CRNCY_CODE,
       hd.TRAN_AMT, hd.PART_TRAN_TYPE, hd.TRAN_TYPE, hd.TRAN_SUB_TYPE, hd.RPT_CODE,
       hd.TRAN_PARTICULAR, hd.TRAN_RMKS, hd.REF_NUM, hd.ENTRY_USER_ID, hd.RCRE_TIME,
       gm.FORACID, gm.ACCT_OPN_DATE
from RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd
left join RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm on hd.ACID = gm.ACID
{FEE_PATTERNS_JOIN}
where hd.ACID = '{FRONTIER_ACID}'
  and hd.RPT_CODE <> 'CHG'
  and N.FEE_TXN IS NULL
order by hd.TRAN_DATE, hd.RCRE_TIME
"""

In [8]:
frontier = pd.read_sql(q1, conn)
frontier.to_csv("01_frontier_ledger.csv", index=False)
print(len(frontier)); frontier.head()

/tmp/ipykernel_3149627/2163817655.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  frontier = pd.read_sql(q1, conn)


20


,TRAN_DATE,TRAN_ID,PART_TRAN_SRL_NUM,TRAN_CRNCY_CODE,TRAN_AMT,PART_TRAN_TYPE,TRAN_TYPE,TRAN_SUB_TYPE,RPT_CODE,TRAN_PARTICULAR,TRAN_RMKS,REF_NUM,ENTRY_USER_ID,RCRE_TIME,FORACID,ACCT_OPN_DATE
0,2025-11-04,0378139,1,USD,100.0,C,C,RI,CMI,ASHIRAF,TEXTILE BUSINESS,None,266331,2025-11-04 16:47:10,9030027282317,2025-11-04
1,2025-11-04,S99264131,2,USD,88.5,C,T,BI,TRF,Ac xfr from gl 0603 to 3500,TACBSH,None,UGCTRLM,2025-11-05 02:04:49,9030027282317,2025-11-04
2,2025-11-04,S99264131,1,USD,88.5,D,T,BI,TRF,Ac xfr from gl 0603 to 3500,TACBSH,None,UGCTRLM,2025-11-05 02:04:49,9030027282317,2025-11-04
3,2026-03-26,S35728440,1,USD,103.0,C,T,CI,TRF,JI,BOL 87318025 KEYSTONE TEXTILES,01,UGCTRLM,2026-03-26 10:30:09,9030027282317,2025-11-04
4,2026-03-30,S37125360,2,USD,99985.0,C,T,CI,TRF,TERENCE A WALTS,IT26089UG0175304,110999594,UGFIVUSR,2026-03-30 19:23:28,9030027282317,2025-11-04


In [9]:
frontier.shape

(20, 16)

In [10]:
q2 = f"""
select hd2.TRAN_DATE, 
hd2.TRAN_ID, 
hd2.PART_TRAN_SRL_NUM,
       hd2.ACID, 
       gm.FORACID, 
       gm.ACCT_NAME, 
       gm.CIF_ID, 
       gm.SOL_ID,
       gm.ACCT_OPN_DATE, 
       gm.SCHM_TYPE, 
       gm.ACCT_CRNCY_CODE,
       hd2.PART_TRAN_TYPE, hd2.TRAN_AMT, hd2.TRAN_SUB_TYPE, hd2.RPT_CODE,
       hd2.TRAN_PARTICULAR, hd2.TRAN_RMKS,
       f.TRAN_AMT as FRONTIER_LEG_AMT, f.PART_TRAN_TYPE as FRONTIER_LEG_TYPE,
       f.TRAN_PARTICULAR as FRONTIER_LEG_PARTICULAR
from RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd2
left join RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm on hd2.ACID = gm.ACID
join RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE f
  on f.TRAN_ID = hd2.TRAN_ID
 and f.TRAN_DATE = hd2.TRAN_DATE
 and f.ACID = '{FRONTIER_ACID}'
 and f.RPT_CODE <> 'CHG'
where hd2.ACID <> '{FRONTIER_ACID}'
order by hd2.TRAN_DATE, hd2.TRAN_ID, hd2.PART_TRAN_SRL_NUM
"""

In [12]:
contra = pd.read_sql(q2, conn)
contra.to_csv("02_frontier_contra_legs.csv", index=False)
print(len(contra)); contra.head(30)

/tmp/ipykernel_3149627/294857198.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contra = pd.read_sql(q2, conn)


7920


,TRAN_DATE,TRAN_ID,PART_TRAN_SRL_NUM,ACID,FORACID,ACCT_NAME,CIF_ID,SOL_ID,ACCT_OPN_DATE,SCHM_TYPE,ACCT_CRNCY_CODE,PART_TRAN_TYPE,TRAN_AMT,TRAN_SUB_TYPE,RPT_CODE,TRAN_PARTICULAR,TRAN_RMKS,FRONTIER_LEG_AMT,FRONTIER_LEG_TYPE,FRONTIER_LEG_PARTICULAR
0,2025-11-04,0378139,2,UG31979,031075USD0103026,TELLER- 26,None,031075,1900-01-01,OAB,USD,D,100.00,RI,None,CASH RECEIPT,None,100.0,C,ASHIRAF
1,2025-11-04,S99211978,1,UG6063,031075USD2817000,DEFAULT PROXY ACCOUNT,None,031075,1900-01-01,OSP,USD,D,10.00,BI,None,PROXY POSTING REQUESTED,Original A/C 9030010621789,10.0,D,FEE - CASH DEPOSIT
2,2025-11-04,S99211978,1,UG6063,031075USD2817000,DEFAULT PROXY ACCOUNT,None,031075,1900-01-01,OSP,USD,D,10.00,BI,None,PROXY POSTING REQUESTED,Original A/C 9030010621789,1.5,D,EXCISE DUTY FEE - CASH DEPOSIT
3,2025-11-04,S99211978,2,UG6063,031075USD2817000,DEFAULT PROXY ACCOUNT,None,031075,1900-01-01,OSP,USD,D,1.50,BI,None,PROXY POSTING REQUESTED,Original A/C 9030010621789,1.5,D,EXCISE DUTY FEE - CASH DEPOSIT
4,2025-11-04,S99211978,2,UG6063,031075USD2817000,DEFAULT PROXY ACCOUNT,None,031075,1900-01-01,OSP,USD,D,1.50,BI,None,PROXY POSTING REQUESTED,Original A/C 9030010621789,10.0,D,FEE - CASH DEPOSIT
5,2025-11-04,S99211978,3,UG658822,031075USD8005006,COMMISSION: CASH DEPOSIT FEE,None,031075,1900-01-01,OAB,USD,C,10.00,BI,None,FEE - CASH DEPOSIT,FEE - CASH DEPOSIT,10.0,D,FEE - CASH DEPOSIT
6,2025-11-04,S99211978,3,UG658822,031075USD8005006,COMMISSION: CASH DEPOSIT FEE,None,031075,1900-01-01,OAB,USD,C,10.00,BI,None,FEE - CASH DEPOSIT,FEE - CASH DEPOSIT,1.5,D,EXCISE DUTY FEE - CASH DEPOSIT
7,2025-11-04,S99211978,4,UG2850799,031075USD5215050,EXCISE DUTY PAYABLE,None,031075,2015-03-22,OAB,USD,C,1.50,BI,None,EXCISE DUTY FEE - CASH DEPOSIT,ExciseDuty AccessFee,10.0,D,FEE - CASH DEPOSIT
8,2025-11-04,S99211978,4,UG2850799,031075USD5215050,EXCISE DUTY PAYABLE,None,031075,2015-03-22,OAB,USD,C,1.50,BI,None,EXCISE DUTY FEE - CASH DEPOSIT,ExciseDuty AccessFee,1.5,D,EXCISE DUTY FEE - CASH DEPOSIT
9,2025-11-04,S99211978,5,UG3128330,9030012182391,BEST PETROLEUM COMPANY LIMITED,20301122619,031075,2016-06-23,ODA,USD,D,10.00,BI,TRF,FEE - CASH DEPOSIT,FEE - CASH DEPOSIT,10.0,D,FEE - CASH DEPOSIT


In [17]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [19]:
q_screen = """
select
    gm.ACID, gm.FORACID, gm.ACCT_NAME, gm.CIF_ID, gm.SOL_ID,
    gm.ACCT_OPN_DATE, gm.SCHM_TYPE, gm.ACCT_CRNCY_CODE,
    count(*)                                           as txn_count,
    min(hd.TRAN_DATE)                                  as first_txn_date,
    max(hd.TRAN_DATE)                                  as last_txn_date,
    datediff(day, gm.ACCT_OPN_DATE, min(hd.TRAN_DATE)) as days_to_first_txn,
    sum(case when hd.PART_TRAN_TYPE='C' then hd.TRAN_AMT else 0 end) as total_credits,
    sum(case when hd.PART_TRAN_TYPE='D' then hd.TRAN_AMT else 0 end) as total_debits,
    max(case when hd.PART_TRAN_TYPE='C' then hd.TRAN_AMT else 0 end) as max_single_credit,
    sum(case when hd.TRAN_AMT >= 80000 and hd.TRAN_AMT < 100000 then 1 else 0 end) as near_100k_count,
    sum(case when hd.PART_TRAN_TYPE='C' and hd.TRAN_AMT >= 10000 then hd.TRAN_AMT else 0 end) as credits_ge_10k,
    count(distinct case when hd.PART_TRAN_TYPE='C' and hd.TRAN_AMT >= 10000 then hd.TRAN_DATE end) as large_credit_days,
    count(distinct hd.TRAN_DATE)                       as active_days
from RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm
join RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd on hd.ACID = gm.ACID
where gm.ACCT_OPN_DATE >= '2023-01-01'
  and gm.ACCT_CRNCY_CODE = 'USD'
  and hd.RPT_CODE <> 'CHG'
  and hd.TRAN_AMT >= 100
group by gm.ACID, gm.FORACID, gm.ACCT_NAME, gm.CIF_ID, gm.SOL_ID,
         gm.ACCT_OPN_DATE, gm.SCHM_TYPE, gm.ACCT_CRNCY_CODE
having sum(case when hd.PART_TRAN_TYPE='C' then hd.TRAN_AMT else 0 end) >= 50000
"""
screen = pd.read_sql(q_screen, conn)
print(len(screen), "accounts")
screen.to_csv("04_screening_aggregates_raw.csv", index=False)
screen.head()

/tmp/ipykernel_3149627/841288508.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  screen = pd.read_sql(q_screen, conn)


7753 accounts


,ACID,FORACID,ACCT_NAME,CIF_ID,SOL_ID,ACCT_OPN_DATE,SCHM_TYPE,ACCT_CRNCY_CODE,txn_count,first_txn_date,last_txn_date,days_to_first_txn,total_credits,total_debits,max_single_credit,near_100k_count,credits_ge_10k,large_credit_days,active_days
0,UG5646388,9030021454612,FREDERICK MAHELE MONGA LIHAU,10302271297,031038,2023-01-10,ODA,USD,160,2023-01-13,2026-06-18,3,"190,295.43","174,840.90","150,000.00",0,"189,995.43",2,135
1,UG5686671,9030021813957,BETTIE ATYAM,10300163479,031078,2023-03-13,LAA,USD,64,2023-03-14,2025-12-24,1,"86,248.28","78,654.07","2,725.86",0,0.00,0,36
2,UG5646379,9030021454426,CARL ELECTRONICS CO LIMITED,20301166582,031014,2023-01-10,ODA,USD,7,2023-01-10,2023-04-20,0,"90,307.00","90,075.00","50,000.00",1,"90,000.00",1,4
3,UG5847538,9030023320869,SEDAT GUNES,10302232010,031078,2023-08-22,ODA,USD,174,2023-08-24,2026-07-02,2,"218,768.93","209,569.93","11,500.00",0,"22,000.00",2,123
4,UG5650223,9030021488819,RHONA TWINOMUGISHA,10302273638,031038,2023-01-16,ODA,USD,316,2023-01-16,2026-07-11,0,"307,670.26","291,797.87","54,346.00",0,"128,596.00",6,233


In [20]:
print(screen['SCHM_TYPE'].value_counts())

# Keep the scheme types that match Frontier's (check Frontier's own row first):
print(screen[screen['ACID']==FRONTIER_ACID][['ACCT_NAME','SCHM_TYPE','total_credits','total_debits']])

KEEP_SCHEMES = list(screen.loc[screen['ACID']==FRONTIER_ACID,'SCHM_TYPE'].unique())
# add other corporate current-account codes you recognise, e.g. KEEP_SCHEMES += ['CAA']
s = screen[screen['SCHM_TYPE'].isin(KEEP_SCHEMES)].copy()
print(len(s), "after scheme filter")

SCHM_TYPE
ODA    5073
LAA    2253
TDA     225
SBA     194
CAA       8
Name: count, dtype: int64
                                ACCT_NAME SCHM_TYPE  total_credits  \
5751  FRONTIER TEXTILE DEALERS CO LIMITED       ODA   1,331,062.50   

      total_debits  
5751  1,327,947.18  
5073 after scheme filter


In [21]:
import numpy as np

s['flow_ratio']    = s['total_debits'] / s['total_credits'].replace(0, np.nan)   # ~1 = pass-through
s['burst_ratio']   = s['credits_ge_10k'] / s['total_credits'].replace(0, np.nan) # credit value arriving in large lumps
s['tempo']         = s['credits_ge_10k'] / s['large_credit_days'].replace(0, np.nan)  # avg large-credit value per active day
s['acct_age_days'] = (pd.Timestamp('today') - pd.to_datetime(s['ACCT_OPN_DATE'])).dt.days

# The five flags
s['f_young']      = (s['acct_age_days'] <= 730).astype(int)                       # opened within ~2 yrs
s['f_dormant']    = (s['days_to_first_txn'] >= 60).astype(int)                    # slow start (proxy for dormancy)
s['f_passthru']   = s['flow_ratio'].between(0.85, 1.15).fillna(False).astype(int) # in ≈ out
s['f_near100k']   = (s['near_100k_count'] >= 2).astype(int)                       # structuring band hits
s['f_burst']      = ((s['burst_ratio'] >= 0.7) & (s['large_credit_days'] <= 15)).fillna(False).astype(int)

FLAGS = ['f_young','f_dormant','f_passthru','f_near100k','f_burst']
s['score'] = s[FLAGS].sum(axis=1)

In [22]:
s_ranked = s.sort_values(['score','total_credits'], ascending=[False,False]).reset_index(drop=True)
fr_rank = s_ranked.index[s_ranked['ACID']==FRONTIER_ACID]
print("Frontier rank:", int(fr_rank[0])+1 if len(fr_rank) else "NOT FOUND")
print(s_ranked.loc[s_ranked['ACID']==FRONTIER_ACID, FLAGS + ['score','flow_ratio','near_100k_count']])

Frontier rank: 53
    f_young  f_dormant  f_passthru  f_near100k  f_burst  score  flow_ratio  \
52        1          0           1           1        1      4        1.00   

    near_100k_count  
52                3  


In [23]:
shortlist = s_ranked[s_ranked['score'] >= 3].copy()
print(len(shortlist), "accounts scoring 3+")

cols = ['FORACID','ACCT_NAME','SOL_ID','ACCT_OPN_DATE','score'] + FLAGS + \
       ['total_credits','total_debits','flow_ratio','near_100k_count','max_single_credit','days_to_first_txn']
shortlist[cols].head(30)

1503 accounts scoring 3+


,FORACID,ACCT_NAME,SOL_ID,ACCT_OPN_DATE,score,f_young,f_dormant,f_passthru,f_near100k,f_burst,total_credits,total_debits,flow_ratio,near_100k_count,max_single_credit,days_to_first_txn
0,9030027305422,KEYSTONE TEXTILES LIMITED,031075,2025-11-08,5,1,1,1,1,1,"3,873,367.84","3,871,864.41",1.00,4,"460,747.50",125
1,9030025506420,CABLINKS ESTABLISHMENTS LIMITED,031086,2024-11-04,5,1,1,1,1,1,"1,394,210.50","1,393,334.14",1.00,5,"287,257.50",84
2,9030025626477,VINCENT NAMWANJA,031003,2024-12-03,5,1,1,1,1,1,"972,289.14","966,113.34",0.99,8,"120,000.00",303
3,9030026953838,YOOLA AND TOOLA INVESTMENTS LIMITED,031084,2025-09-01,5,1,1,1,1,1,"698,697.44","698,430.18",1.00,6,"169,540.00",151
4,9030027584488,TOME ADVERTISING LIMITED,031003,2026-01-13,5,1,1,1,1,1,"537,100.00","535,868.48",1.00,2,"90,000.00",63
5,9030027421767,CHILDREN OF GRACE UGANDA LTD,031026,2025-12-03,5,1,1,1,1,1,"300,175.00","284,000.00",0.95,2,"95,025.00",72
6,9030026098202,REEF INDUSTRIES LTD,031003,2025-03-14,5,1,1,1,1,1,"263,912.06","263,227.49",1.00,2,"80,539.20",61
7,9030026690675,UGANDA NATIONAL OIL COMPANY LIMITED,031002,2025-07-17,4,1,1,1,0,1,"160,676,826.35","160,076,826.35",1.00,0,"52,000,000.00",217
8,9030027966089,CS HOSPITALITY LIMITED - COLLECTION ACCOUNT,031006,2026-03-24,4,1,1,1,0,1,"54,768,021.09","54,768,021.09",1.00,0,"50,000,000.00",97
9,9030025638750,GAOLOU COMPANY LTD,031078,2024-12-06,4,1,1,1,1,0,"25,209,475.19","25,164,060.53",1.00,80,"350,000.00",61


In [24]:
# 1. Score distribution — how big is each tier?
print(s_ranked['score'].value_counts().sort_index(ascending=False))

# 2. How many 3+ (the shortlist)?
print("shortlist size:", (s_ranked['score']>=3).sum())

# 3. THE KEY VIEW — everything ranked at or above Frontier, plus all score-5s:
top = s_ranked.head(80)
print(top[['ACCT_NAME','SOL_ID','ACCT_OPN_DATE','score','flow_ratio',
           'near_100k_count','total_credits','days_to_first_txn']].to_string())

# 4. Branch concentration among high scorers vs the population
print(s_ranked[s_ranked['score']>=4]['SOL_ID'].value_counts().head(10))
print(s_ranked['SOL_ID'].value_counts().head(10))

score
5       7
4     224
3    1272
2    1875
1    1513
0     182
Name: count, dtype: int64
shortlist size: 1503
                                                   ACCT_NAME  SOL_ID ACCT_OPN_DATE  score  flow_ratio  near_100k_count  total_credits  days_to_first_txn
0                                  KEYSTONE TEXTILES LIMITED  031075    2025-11-08      5        1.00                4   3,873,367.84                125
1                            CABLINKS ESTABLISHMENTS LIMITED  031086    2024-11-04      5        1.00                5   1,394,210.50                 84
2                                           VINCENT NAMWANJA  031003    2024-12-03      5        0.99                8     972,289.14                303
3                        YOOLA AND TOOLA INVESTMENTS LIMITED  031084    2025-09-01      5        1.00                6     698,697.44                151
4                                   TOME ADVERTISING LIMITED  031003    2026-01-13      5        1.00                2    